# Extension 3: Sequential Unlearning — GA vs PO

**Goal:** Measure metric degradation as 20 forget-set IDs are unlearned sequentially in 3 rounds using 2 contrasting methods (GA, PO).

**Design:**
- Round 1: Unlearn batch 1 (7 IDs) from stage1
- Round 2: Unlearn batch 2 (7 IDs) from round1 checkpoint
- Round 3: Unlearn batch 3 (6 IDs) from round2 checkpoint

**Output:** Metrics table showing rouge_l, exact_match, mia_mink, ks_test_pval, avg_mu across rounds 

## Section 0: Setup

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

# Skip clone if repo already exists and is valid
if not Path(PROJECT_ROOT).exists() or not (Path(PROJECT_ROOT) / 'FIUBench').exists():
    print("Cloning FIUBench_Reproducing...\n")
    
    # Use public clone (or set GH_TOKEN env var if private)
    clone_url = "https://github.com/akashyall34/FIUBench_Reproducing.git"
    
    result = subprocess.run(
        f"git clone {clone_url} {PROJECT_ROOT}",
        shell=True, capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✅ Repository cloned successfully")
    else:
        print("❌ Clone failed:")
        print(result.stderr)
        raise RuntimeError(f"Failed to clone repo")
else:
    print(f"✅ Repository already exists at {PROJECT_ROOT}")

# Verify structure
assert (Path(PROJECT_ROOT) / 'FIUBench').exists(), "❌ FIUBench subdirectory not found"
print(f"✅ FIUBench directory confirmed")

✅ Repository already exists at /content/FIUBench_Reproducing
✅ FIUBench directory confirmed


In [9]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

# Stash local changes first to avoid conflicts
print("Stashing local changes...")
result = subprocess.run(
    "git stash",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)
if result.returncode == 0 and result.stdout.strip():
    print(result.stdout)
else:
    print("(no local changes to stash)")

# Now pull
result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

Stashing local changes...
No local changes to save

✅ Repository updated
Already up to date.



In [10]:
import subprocess, sys

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

import transformers
import torch
print(f"✅ torch={torch.__version__}  transformers={transformers.__version__}")

✅ torch=2.4.1+cu121  transformers=4.48.0


## Copy Stage 1

In [11]:
import os, sys, re
from pathlib import Path

# ── Setup paths ───────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = '/content/FIUBench_Reproducing' if IN_COLAB else '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/FIUBench_Reproducing'
STAGE1_LOCAL = '/content/stage1_final' if IN_COLAB else f'{PROJECT_ROOT}/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'

os.environ['WANDB_MODE'] = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

# ── 1. Copy Stage 1 from Drive ─────────────────────────────────────────────────
if IN_COLAB:
    if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
        print("Copying Stage 1 from Drive...")
        Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
        ret = os.system(f'rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
        assert ret == 0, "rsync failed"

safetensors = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert safetensors, f"No safetensors in {STAGE1_LOCAL}"
print(f"✅ Stage 1 ready: {[f.name for f in safetensors]}")

# ── 2. Patch forget.py ─────────────────────────────────────────────────────────
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
assert fg_py.exists(), f"❌ forget.py not found at {fg_py}"

src = fg_py.read_text()
patched = src
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py")
else:
    print("✅ forget.py already patched")

assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "bfloat16 patch missing"
assert 'mixed_precision="bf16"' in fg_py.read_text(), "mixed_precision patch missing"

# ── 3. Patch modeling_llava.py ─────────────────────────────────────────────────
import glob
llava_candidates = glob.glob(
    '/usr/local/lib/python*/dist-packages/transformers/models/llava/modeling_llava.py'
)
if llava_candidates:
    llava_path = llava_candidates[0]
    llava_src = Path(llava_path).read_text()
    llava_patched = re.sub(
        r'n_image_tokens != n_image_features',
        'False',
        llava_src
    )
    if llava_patched != llava_src:
        Path(llava_path).write_text(llava_patched)
        print(f"✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")
else:
    print("⚠️  modeling_llava.py not found — may not be needed for this transformers version")

print(f"\n✅ Setup complete:")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   FIUBENCH_DIR: {FIUBENCH_DIR}")
print(f"   STAGE1_LOCAL: {STAGE1_LOCAL}")
print(f"\n✅ All patches applied. Ready for Stage 2.")

Copying Stage 1 from Drive...
✅ Stage 1 ready: ['model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']
✅ forget.py already patched
✅ Patched modeling_llava.py

✅ Setup complete:
   PROJECT_ROOT: /content/FIUBench_Reproducing
   FIUBENCH_DIR: /content/FIUBench_Reproducing/FIUBench
   STAGE1_LOCAL: /content/stage1_final

✅ All patches applied. Ready for Stage 2.


## Section 1: Load & Partition Forget Set

In [12]:
import json
from pathlib import Path

# Load full dataset
with open(f'{FIUBENCH_DIR}/dataset/full.json') as f:
    full_data = [json.loads(line) for line in f if line.strip()]
with open(f'{FIUBENCH_DIR}/dataset/split.json') as f:
    splits = json.load(f)

forget5_ids = list(set(splits['forget5']))
forget_data = [d for d in full_data if d['unique_id'] in forget5_ids]

print(f'Total forget5 identities: {len(forget_data)}')

# Shuffle and split into 3 batches
np.random.seed(42)
shuffled_forget = forget_data.copy()
np.random.shuffle(shuffled_forget)

batch1 = shuffled_forget[0:7]   # 7 IDs
batch2 = shuffled_forget[7:14]  # 7 IDs
batch3 = shuffled_forget[14:20] # 6 IDs

print(f'Batch 1: {len(batch1)} IDs')
print(f'Batch 2: {len(batch2)} IDs')
print(f'Batch 3: {len(batch3)} IDs')
print(f'Total: {len(batch1) + len(batch2) + len(batch3)} IDs')

Total forget5 identities: 20
Batch 1: 7 IDs
Batch 2: 7 IDs
Batch 3: 6 IDs
Total: 20 IDs


## Section 2: Restore Stage1 & Apply Patches

In [13]:
import os
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final' if IN_COLAB else f'{PROJECT_ROOT}/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'

# Verify stage1 is ready (Cell-7 already copied it from Drive)
stage1_files = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert stage1_files, f'No safetensors in {STAGE1_LOCAL}'
print(f'✅ Stage1 ready: {len(stage1_files)} files')

# Verify forget.py patches are in place (Cell-7 already applied them)
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "❌ bfloat16 patch missing"
assert 'mixed_precision="bf16"' in fg_py.read_text(), "❌ mixed_precision patch missing"
print('✅ forget.py patches verified')
print('\n✅ All prerequisites ready for sequential unlearning')

✅ Stage1 ready: 2 files
✅ forget.py patches verified

✅ All prerequisites ready for sequential unlearning


## Dataset

In [20]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

PROJECT_ROOT = '/content/FIUBench_Reproducing'

try:
    sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
    sfhq_dir.mkdir(parents=True, exist_ok=True)

    existing = list(sfhq_dir.glob("*.jpg"))
    if len(existing) >= 400:
        print(f"✅ SFHQ images already present: {len(existing)}")
    else:
        print("📥 Downloading SFHQ.zip from Hugging Face...")
        zip_path = hf_hub_download(
            repo_id="gray311/FIUBench",
            filename="SFHQ.zip",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )

        extract_dir = sfhq_dir.parent / "sfhq_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"📦 Extracting SFHQ.zip...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        found = list(extract_dir.rglob("*.jpg"))
        print(f"Found {len(found)} jpg files")

        copied = 0
        for src in found:
            dst = sfhq_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1

        total = len(list(sfhq_dir.glob("*.jpg")))
        print(f"✅ SFHQ ready: {total} images")

except Exception as e:
    print(f"❌ SFHQ download failed: {e}")
    raise

# Create symlink in FIUBENCH_DIR
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = Path(FIUBENCH_DIR) / "dataset" / "SFHQ"

if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
print(f"✅ Symlinked: {n} images")


📥 Downloading SFHQ.zip from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

📦 Extracting SFHQ.zip...
Found 1000 jpg files
✅ SFHQ ready: 1000 images
✅ Symlinked: 1000 images


In [21]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

# Verify source exists
if not sfhq_src.exists():
    print(f"❌ Source not found: {sfhq_src}")
    raise FileNotFoundError(f"SFHQ images not at {sfhq_src}")

# Clean up old symlink/directory
if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

# Create symlink
sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
if n < 400:
    print(f"⚠️  Only {n} images (expected 400+)")
else:
    print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [22]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## GA

In [24]:
import subprocess, time, json, os
from pathlib import Path

METHOD = 'ga'
LR = '2e-5'
PORT = '29500'

results_ga = {}

# ── ROUND 1 ───────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'[GA] ROUND 1: Batch 1 (7 IDs) from stage1')
print('='*70)

stage2_r1 = Path(f'/content/ext3_ga_r1') if IN_COLAB else Path(f'/tmp/ext3_ga_r1')
stage2_r1.mkdir(parents=True, exist_ok=True)

cmd_r1 = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={stage2_r1} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true"
)

print(f'Training...')
t0 = time.time()
proc = subprocess.Popen(cmd_r1, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode
elapsed = int(time.time() - t0)

if ret == 0:
    print(f'\n✅ Round 1 complete ({elapsed}s)')
else:
    print(f'\n❌ Round 1 failed (exit {ret})')

results_ga[1] = {'cumulative_ids': 7, 'model_path': str(stage2_r1)}

# ── ROUND 2 ───────────────────────────────────────────────────────────────────
if ret == 0:
    print('\n' + '='*70)
    print(f'[GA] ROUND 2: Batch 2 (7 IDs) from round1 checkpoint')
    print('='*70)

    stage2_r2 = Path(f'/content/ext3_ga_r2') if IN_COLAB else Path(f'/tmp/ext3_ga_r2')
    stage2_r2.mkdir(parents=True, exist_ok=True)

    cmd_r2 = (
        f"cd {FIUBENCH_DIR} && "
        f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
        f"--config-name forget_lora "
        f"model_path={stage2_r1} "
        f"save_dir={stage2_r2} "
        f"split=forget5 "
        f"forget_loss={METHOD} "
        f"lr={LR} "
        f"batch_size=8 "
        f"num_epochs=8 "
        f"seed=233 "
        f"overwrite_dir=true"
    )

    print(f'Training...')
    t0 = time.time()
    proc = subprocess.Popen(cmd_r2, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    ret = proc.returncode
    elapsed = int(time.time() - t0)

    if ret == 0:
        print(f'\n✅ Round 2 complete ({elapsed}s)')
    else:
        print(f'\n❌ Round 2 failed (exit {ret})')

    results_ga[2] = {'cumulative_ids': 14, 'model_path': str(stage2_r2)}

    # ── ROUND 3 ─────────────────────────────────────────────────────────────────
    if ret == 0:
        print('\n' + '='*70)
        print(f'[GA] ROUND 3: Batch 3 (6 IDs) from round2 checkpoint')
        print('='*70)

        stage2_r3 = Path(f'/content/ext3_ga_r3') if IN_COLAB else Path(f'/tmp/ext3_ga_r3')
        stage2_r3.mkdir(parents=True, exist_ok=True)

        cmd_r3 = (
            f"cd {FIUBENCH_DIR} && "
            f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
            f"--config-name forget_lora "
            f"model_path={stage2_r2} "
            f"save_dir={stage2_r3} "
            f"split=forget5 "
            f"forget_loss={METHOD} "
            f"lr={LR} "
            f"batch_size=8 "
            f"num_epochs=8 "
            f"seed=233 "
            f"overwrite_dir=true"
        )

        print(f'Training...')
        t0 = time.time()
        proc = subprocess.Popen(cmd_r3, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        ret = proc.returncode
        elapsed = int(time.time() - t0)

        if ret == 0:
            print(f'\n✅ Round 3 complete ({elapsed}s)')
        else:
            print(f'\n❌ Round 3 failed (exit {ret})')

        results_ga[3] = {'cumulative_ids': 20, 'model_path': str(stage2_r3)}

print(f'\n✅ GA sequential unlearning complete')


[GA] ROUND 1: Batch 1 (7 IDs) from stage1
Training...
2026-04-27 18:47:56.849365: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 18:47:56.867677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777315676.889217   16517 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777315676.895844   16517 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777315676.912623   16517 computation_placer.cc:177] computation placer alre

## Section 4: Sequential Unlearning — PO

In [ ]:
import subprocess, time, json, os
from pathlib import Path

METHOD = 'po'
METHOD_LABEL = 'idk'  # PO uses 'idk' as forget_loss
LR = '3e-4'
PORT = '29503'

results_po = {}

# ── ROUND 1 ───────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'[PO] ROUND 1: Batch 1 (7 IDs) from stage1')
print('='*70)

stage2_r1 = Path(f'/content/ext3_po_r1') if IN_COLAB else Path(f'/tmp/ext3_po_r1')
stage2_r1.mkdir(parents=True, exist_ok=True)

cmd_r1 = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={stage2_r1} "
    f"split=forget5 "
    f"forget_loss={METHOD_LABEL} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true"
)

print(f'Training...')
t0 = time.time()
proc = subprocess.Popen(cmd_r1, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode
elapsed = int(time.time() - t0)

if ret == 0:
    print(f'\n✅ Round 1 complete ({elapsed}s)')
else:
    print(f'\n❌ Round 1 failed (exit {ret})')

results_po[1] = {'cumulative_ids': 7, 'model_path': str(stage2_r1), 'base_model': STAGE1_LOCAL}

# ── ROUND 2 ───────────────────────────────────────────────────────────────────
if ret == 0:
    print('\n' + '='*70)
    print(f'[PO] ROUND 2: Batch 2 (7 IDs) from round1 checkpoint')
    print('='*70)

    stage2_r2 = Path(f'/content/ext3_po_r2') if IN_COLAB else Path(f'/tmp/ext3_po_r2')
    stage2_r2.mkdir(parents=True, exist_ok=True)
    
    # Merge LoRA from Round 1
    print('\n📦 Preparing Round 2 input model (merging LoRA from Round 1)...')
    merged_r1_path = Path(f'/content/ext3_po_r1_merged') if IN_COLAB else Path(f'/tmp/ext3_po_r1_merged')
    merged_r1_path.mkdir(parents=True, exist_ok=True)
    
    try:
        merge_lora_checkpoint(str(stage2_r1), STAGE1_LOCAL, str(merged_r1_path))
        r1_model_path = str(merged_r1_path)
        print(f'✅ Merged model ready at {r1_model_path}')
    except Exception as e:
        print(f'⚠️  Merge failed: {e}, using stage1 instead')
        r1_model_path = STAGE1_LOCAL

    cmd_r2 = (
        f"cd {FIUBENCH_DIR} && "
        f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
        f"--config-name forget_lora "
        f"model_path={r1_model_path} "
        f"save_dir={stage2_r2} "
        f"split=forget5 "
        f"forget_loss={METHOD_LABEL} "
        f"lr={LR} "
        f"batch_size=8 "
        f"num_epochs=8 "
        f"seed=233 "
        f"overwrite_dir=true"
    )

    print(f'Training...')
    t0 = time.time()
    proc = subprocess.Popen(cmd_r2, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    ret = proc.returncode
    elapsed = int(time.time() - t0)

    if ret == 0:
        print(f'\n✅ Round 2 complete ({elapsed}s)')
    else:
        print(f'\n❌ Round 2 failed (exit {ret})')

    results_po[2] = {'cumulative_ids': 14, 'model_path': str(stage2_r2), 'base_model': STAGE1_LOCAL}

    # ── ROUND 3 ─────────────────────────────────────────────────────────────────
    if ret == 0:
        print('\n' + '='*70)
        print(f'[PO] ROUND 3: Batch 3 (6 IDs) from round2 checkpoint')
        print('='*70)

        stage2_r3 = Path(f'/content/ext3_po_r3') if IN_COLAB else Path(f'/tmp/ext3_po_r3')
        stage2_r3.mkdir(parents=True, exist_ok=True)
        
        # Merge LoRA from Round 2
        print('\n📦 Preparing Round 3 input model (merging LoRA from Round 2)...')
        merged_r2_path = Path(f'/content/ext3_po_r2_merged') if IN_COLAB else Path(f'/tmp/ext3_po_r2_merged')
        merged_r2_path.mkdir(parents=True, exist_ok=True)
        
        try:
            merge_lora_checkpoint(str(stage2_r2), STAGE1_LOCAL, str(merged_r2_path))
            r2_model_path = str(merged_r2_path)
            print(f'✅ Merged model ready at {r2_model_path}')
        except Exception as e:
            print(f'⚠️  Merge failed: {e}, using stage1 instead')
            r2_model_path = STAGE1_LOCAL

        cmd_r3 = (
            f"cd {FIUBENCH_DIR} && "
            f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
            f"--config-name forget_lora "
            f"model_path={r2_model_path} "
            f"save_dir={stage2_r3} "
            f"split=forget5 "
            f"forget_loss={METHOD_LABEL} "
            f"lr={LR} "
            f"batch_size=8 "
            f"num_epochs=8 "
            f"seed=233 "
            f"overwrite_dir=true"
        )

        print(f'Training...')
        t0 = time.time()
        proc = subprocess.Popen(cmd_r3, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        ret = proc.returncode
        elapsed = int(time.time() - t0)

        if ret == 0:
            print(f'\n✅ Round 3 complete ({elapsed}s)')
        else:
            print(f'\n❌ Round 3 failed (exit {ret})')

        results_po[3] = {'cumulative_ids': 20, 'model_path': str(stage2_r3), 'base_model': STAGE1_LOCAL}

print(f'\n✅ PO sequential unlearning complete')

## Section 5: Evaluate Each Checkpoint

Run eval_accurate_*.py for each round checkpoint and record metrics.

In [ ]:
import subprocess, json, yaml, os
from pathlib import Path
import pandas as pd

print('\n' + '='*70)
print('EVALUATION PHASE — Running Evaluations')
print('='*70)

metrics_results = {'ga': {}, 'po': {}}

# Helper to run evaluation and extract metrics
def evaluate_checkpoint(method, round_num, model_path, cumulative_ids):
    print(f'\n[{method.upper()}] Round {round_num}: Evaluating {cumulative_ids} cumulative IDs...')

    save_dir = Path(model_path) / 'eval_results'
    save_dir.mkdir(parents=True, exist_ok=True)

    cmd = (
        f"cd {FIUBENCH_DIR} && "
        f"python evaluate_util.py --config-name eval.yaml "
        f"model_path={model_path} "
        f"model_family=llava-phi "
        f"save_dir={save_dir} "
        f"LoRA.lora_path={model_path}/checkpoint.pt "
        f"overwrite=true"
    )

    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line.lower() for k in ['rouge', 'exact', 'mia', 'ks_test', 'avg']):
            print(f'  {line.strip()}')
    proc.wait()

    if proc.returncode == 0:
        print(f'  ✅ Evaluation complete')
        return True
    else:
        print(f'  ❌ Evaluation failed (exit {proc.returncode})')
        return False

# Evaluate GA checkpoints
for r in sorted(results_ga.keys()):
    model_path = results_ga[r]['model_path']
    cumulative_ids = results_ga[r]['cumulative_ids']
    evaluate_checkpoint('ga', r, model_path, cumulative_ids)
    metrics_results['ga'][r] = {'ids': cumulative_ids, 'path': model_path}

# Evaluate PO checkpoints
for r in sorted(results_po.keys()):
    model_path = results_po[r]['model_path']
    cumulative_ids = results_po[r]['cumulative_ids']
    evaluate_checkpoint('po', r, model_path, cumulative_ids)
    metrics_results['po'][r] = {'ids': cumulative_ids, 'path': model_path}

print('\n' + '='*70)
print('RESULTS SUMMARY')
print('='*70)

results_summary = {
    'ga_rounds': {r: {'cumulative_ids': results_ga[r]['cumulative_ids'],
                       'model_path': str(results_ga[r]['model_path'])}
                  for r in sorted(results_ga.keys())},
    'po_rounds': {r: {'cumulative_ids': results_po[r]['cumulative_ids'],
                       'model_path': str(results_po[r]['model_path'])}
                  for r in sorted(results_po.keys())}
}

print(json.dumps(results_summary, indent=2))

# Save results to file
results_path = Path(FIUBENCH_DIR).parent / 'results' / 'extension3_results.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'\n✅ Results saved to {results_path}')